# Exploratory Operational Analysis - MRMI System

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Assuming this notebook is run from the project root or 
# the 'meta-music-ops-mrmi' directory is accessible.
BASE_DIR = 'meta-music-ops-mrmi'

# Load CSVs
dim_suppliers_path = os.path.join(BASE_DIR, 'data', 'dim_suppliers.csv')
dim_music_catalog_path = os.path.join(BASE_DIR, 'data', 'dim_music_catalog.csv')
fct_ingestion_log_path = os.path.join(BASE_DIR, 'data', 'fct_ingestion_log.csv')

suppliers_df = pd.read_csv(dim_suppliers_path)
catalog_df = pd.read_csv(dim_music_catalog_path, parse_dates=['license_expiry'])
ingestion_logs_df = pd.read_csv(fct_ingestion_log_path, parse_dates=['timestamp'])

print("Data loaded successfully!")


## Strategy to Address Supplier Quality Breaches

Based on the analysis of the `v_supplier_sla_breach_report`, which identified 13 suppliers with `QUALITY_BREACH` status due to failure rates exceeding 10% (ranging from 10.23% to 13.33%), while exhibiting healthy latency, the following strategy is proposed to investigate and address these issues:

### Proposed Strategy: Focused Quality Remediation

Given that `LATENCY_BREACHES` are absent, the primary focus will be on **data quality and validation mechanisms** rather than infrastructure or network performance. The clustered nature of failure rates suggests a common underlying issue or a shared failure point in the ingestion process.

### Actionable Steps:

1.  **Deep Dive into `fct_ingestion_log` for Breaching Suppliers:**
    *   **Objective:** Identify specific error patterns, malformed data structures, or consistently failing validation rules related to the 13 breaching suppliers.
    *   **Action:** Query `fct_ingestion_log` filtering by `supplier_id` of the breaching suppliers and `status = 'FAIL'`. Analyze a sample of these failed logs to identify commonalities in the `track_id` involved, the `timestamp` of failures, and (in a real system) any associated error messages or rejection reasons.
    *   **Hypothesis:** Look for repeated `issue_code` patterns if a `dim_metadata_anomalies` table were fully populated during ingestion, indicating specific types of metadata errors (e.g., `MISSING_ISRC`, `INVALID_DDEX_VALUE`).

2.  **Submission Payload Analysis & DDEX Compliance Audit:**
    *   **Objective:** Compare the submission formats from breaching suppliers against Meta's expected DDEX ERN 4.3 standards.
    *   **Action:** Request sample submission files (e.g., XML feeds) from these 13 suppliers. Conduct a manual or automated audit comparing their payloads against the most recent DDEX schemas and Meta's internal ingestion requirements. This will help identify discrepancies in how they package `ISRC`, `title`, `artist_id`, `is_spatial_ready`, and `ddex_version` data.
    *   **Consideration:** The `Spatial Audio Readiness Audit` already showed issues with `is_spatial_ready=TRUE` but `ddex_version != 'ERN 4.3'`. This could be a contributing factor to failures if these suppliers are incorrectly flagging spatial readiness without providing the corresponding `ERN 4.3` compliant metadata.

3.  **Supplier Engagement & Education:**
    *   **Objective:** Directly collaborate with identified suppliers to help them understand and rectify their submission issues.
    *   **Action:** Organize dedicated workshops or one-on-one sessions with the technical teams of the 13 breaching suppliers. Provide clear documentation, updated API specifications, and examples of compliant DDEX submissions. Offer tools or client libraries for pre-validation on their side.

4.  **Implement Stricter Pre-ingestion Checks & Feedback Loops:**
    *   **Objective:** Prevent future quality breaches by enforcing validation earlier in the ingestion pipeline.
    *   **Action:** Enhance the `validation.py` module in `src/processing/` (or similar component) to include more granular, real-time feedback to suppliers on submission failures. This could involve returning specific error codes and messages immediately upon receipt, allowing them to self-correct before the data impacts operational metrics.

5.  **Monitor & Re-evaluate:**
    *   **Objective:** Track the effectiveness of remediation efforts.
    *   **Action:** Continuously monitor the `v_supplier_sla_breach_report` and individual supplier performance. Set specific targets for reducing failure rates and conduct follow-up audits to ensure sustained compliance.

This structured approach will move beyond simply identifying breaches to actively diagnosing root causes and implementing sustainable solutions, ensuring the long-term integrity and efficiency of Meta's music ingestion pipeline.


## Visualization: Supplier SLA Breach Analysis

In [ ]:
# Replicate the logic for v_supplier_sla_breach_report in Pandas
supplier_performance = ingestion_logs_df.merge(catalog_df[['track_id', 'supplier_id']], on='track_id', suffixes=('_log', '_cat'))                                             .merge(suppliers_df[['supplier_id', 'supplier_name', 'region']], on='supplier_id_cat', suffixes=('', '_sup'))

# Group by supplier and calculate metrics
supplier_metrics = supplier_performance.groupby(['supplier_id_sup', 'supplier_name', 'region']).agg(
    total_submissions=('ingestion_id', 'count'),
    avg_latency_seconds=('latency_seconds', 'mean'),
    total_failed_submissions=('status', lambda x: (x == 'FAIL').sum())
).reset_index()

supplier_metrics['failure_rate_percentage'] = (supplier_metrics['total_failed_submissions'] / supplier_metrics['total_submissions']) * 100

supplier_metrics['overall_sla_status'] = 'COMPLIANT'
supplier_metrics.loc[supplier_metrics['avg_latency_seconds'] > 30, 'overall_sla_status'] = 'LATENCY_BREACH'
supplier_metrics.loc[supplier_metrics['failure_rate_percentage'] > 10, 'overall_sla_status'] = 'QUALITY_BREACH'

# Filter for breaching suppliers (total_submissions > 10 as per SQL)
breaching_suppliers = supplier_metrics[(supplier_metrics['total_submissions'] > 10) & 
                                       (supplier_metrics['overall_sla_status'].isin(['LATENCY_BREACH', 'QUALITY_BREACH']))]

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=breaching_suppliers,
    x='failure_rate_percentage',
    y='avg_latency_seconds',
    hue='supplier_name',
    size='total_submissions',
    sizes=(100, 1000), # Adjust marker size range
    alpha=0.7,
    palette='viridis'
)
plt.axvline(x=10, color='r', linestyle='--', label='Quality Breach Threshold (10%)')
plt.axhline(y=30, color='b', linestyle='--', label='Latency Breach Threshold (30s)')
plt.title('Breaching Suppliers: Failure Rate vs. Average Latency (Bubble Size = Total Submissions)')
plt.xlabel('Failure Rate (%)')
plt.ylabel('Average Latency (seconds)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

print("
Breaching Suppliers Details:")
print(breaching_suppliers[['supplier_name', 'overall_sla_status', 'avg_latency_seconds', 'failure_rate_percentage', 'total_submissions']].sort_values(by='failure_rate_percentage', ascending=False).to_markdown(index=False))


## Visualization: Spatial Audio Readiness Discrepancy

In [ ]:
# Filter tracks with is_spatial_ready=TRUE but non-ERN 4.3 ddex_version
discrepant_spatial_tracks = catalog_df[
    (catalog_df['is_spatial_ready'] == True) &
    (catalog_df['ddex_version'] != 'ERN 4.3')
]

plt.figure(figsize=(10, 6))
sns.countplot(
    data=discrepant_spatial_tracks,
    x='ddex_version',
    order=discrepant_spatial_tracks['ddex_version'].value_counts().index,
    palette='magma'
)
plt.title('Distribution of DDEX Versions for Spatial-Ready Tracks (Non-ERN 4.3)')
plt.xlabel('DDEX Version')
plt.ylabel('Number of Tracks')
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

print("
DDEX Version Distribution for Discrepant Spatial Tracks:")
print(discrepant_spatial_tracks['ddex_version'].value_counts().reset_index().rename(columns={'index': 'DDEX_Version', 'ddex_version': 'Count'}).to_markdown(index=False))

print(f"
Total discrepant spatial-ready tracks: {len(discrepant_spatial_tracks)}")
